# =============================================================
DSU Aerospace - Airfoil ML Prediction (Phase 2)
NACA Airfoil: Cd Prediction using Linear Regression & Random Forest
All airfoils at Re = 1.0 x 10^6, Mach = 0.10, Ncrit = 9.0
Compatible with Google Colab
=============================================================

### CELL 1: Install & Import

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import GroupShuffleSplit
import warnings
warnings.filterwarnings('ignore')

print("All libraries loaded successfully.")
print("Reynolds number: 1.0 x 10^6 for ALL airfoils (fair comparison)")

### CELL 2: Build Dataset from XFLR Polar Data

In [ ]:
# Source: XFLR5 v6.62 exports
# All polars: Re = 1.000e6, Mach = 0.100, Ncrit = 9.0
# Format per row: (alpha, Cl, Cd, Cm)

raw_data = {
    'NACA_64A210': {
        're': 1.0e6,
        'polars': [
            (-4.0, -0.2340, 0.00886, -0.0464),
            (-3.5, -0.1820, 0.00823, -0.0456),
            (-3.0, -0.1293, 0.00784, -0.0450),
            (-2.5, -0.0747, 0.00755, -0.0448),
            (-2.0, -0.0248, 0.00654, -0.0442),
            (-1.5,  0.0198, 0.00517, -0.0429),
            (-1.0,  0.0734, 0.00497, -0.0427),
            (-0.5,  0.1265, 0.00483, -0.0423),
            ( 0.0,  0.1791, 0.00485, -0.0416),
            ( 0.5,  0.2314, 0.00491, -0.0410),
            ( 1.0,  0.2841, 0.00494, -0.0405),
            ( 1.5,  0.3363, 0.00508, -0.0399),
            ( 2.0,  0.3865, 0.00538, -0.0390),
            ( 3.0,  0.4692, 0.00835, -0.0349),
            ( 3.5,  0.5439, 0.00897, -0.0395),
            ( 4.0,  0.6114, 0.00952, -0.0424),
            ( 4.5,  0.6567, 0.01013, -0.0405),
            ( 5.0,  0.7006, 0.01090, -0.0383),
            ( 5.5,  0.7448, 0.01190, -0.0362),
            ( 6.0,  0.7956, 0.01216, -0.0362),
            ( 6.5,  0.8398, 0.01341, -0.0346),
            ( 7.0,  0.8913, 0.01374, -0.0344),
            ( 7.5,  0.9411, 0.01430, -0.0339),
            ( 8.0,  0.9903, 0.01493, -0.0335),
            ( 8.5,  1.0308, 0.01680, -0.0314),
        ]
    },
    'NACA_2412': {
        're': 1.0e6,
        'polars': [
            (-4.0, -0.1981, 0.00774, -0.0558),
            (-3.5, -0.1431, 0.00739, -0.0554),
            (-3.0, -0.0880, 0.00710, -0.0550),
            (-2.5, -0.0329, 0.00682, -0.0546),
            (-2.0,  0.0222, 0.00660, -0.0543),
            (-1.5,  0.0769, 0.00637, -0.0539),
            (-1.0,  0.1311, 0.00611, -0.0535),
            (-0.5,  0.1849, 0.00583, -0.0529),
            ( 0.0,  0.2386, 0.00564, -0.0523),
            ( 0.5,  0.2919, 0.00553, -0.0514),
            ( 1.0,  0.3438, 0.00547, -0.0502),
            ( 1.5,  0.3951, 0.00555, -0.0486),
            ( 2.0,  0.4526, 0.00576, -0.0484),
            ( 2.5,  0.5213, 0.00603, -0.0510),
            ( 3.0,  0.5949, 0.00632, -0.0548),
            ( 3.5,  0.6712, 0.00666, -0.0593),
            ( 4.0,  0.7193, 0.00694, -0.0577),
            ( 4.5,  0.7673, 0.00728, -0.0560),
            ( 5.0,  0.8142, 0.00777, -0.0542),
            ( 5.5,  0.8605, 0.00838, -0.0523),
            ( 6.0,  0.9068, 0.00911, -0.0506),
            ( 6.5,  0.9530, 0.00994, -0.0489),
            ( 7.0,  0.9993, 0.01080, -0.0472),
            ( 7.5,  1.0463, 0.01161, -0.0457),
            ( 8.0,  1.0928, 0.01245, -0.0442),
            ( 8.5,  1.1387, 0.01331, -0.0426),
            ( 9.0,  1.1850, 0.01409, -0.0410),
            ( 9.5,  1.2301, 0.01491, -0.0393),
            (10.0,  1.2719, 0.01591, -0.0371),
            (10.5,  1.3120, 0.01694, -0.0347),
            (11.0,  1.3511, 0.01788, -0.0322),
            (11.5,  1.3805, 0.01913, -0.0281),
            (12.0,  1.3973, 0.02120, -0.0228),
            (12.5,  1.4289, 0.02244, -0.0199),
            (13.0,  1.4554, 0.02410, -0.0168),
            (13.5,  1.4782, 0.02616, -0.0138),
            (14.0,  1.4961, 0.02874, -0.0111),
        ]
    },
    'NACA_4412': {
        're': 1.0e6,
        'polars': [
            (-4.0,  0.0310, 0.00804, -0.1052),
            (-3.5,  0.0872, 0.00781, -0.1051),
            (-3.0,  0.1431, 0.00762, -0.1049),
            (-2.0,  0.2550, 0.00731, -0.1046),
            (-1.5,  0.3108, 0.00720, -0.1044),
            (-1.0,  0.3661, 0.00714, -0.1042),
            (-0.5,  0.4215, 0.00707, -0.1041),
            ( 0.0,  0.4767, 0.00693, -0.1039),
            ( 0.5,  0.5297, 0.00644, -0.1036),
            ( 1.0,  0.5771, 0.00598, -0.1015),
            ( 1.5,  0.6481, 0.00605, -0.1044),
            ( 2.0,  0.7017, 0.00627, -0.1038),
            ( 2.5,  0.7557, 0.00649, -0.1033),
            ( 3.0,  0.8098, 0.00674, -0.1029),
            ( 3.5,  0.8641, 0.00699, -0.1025),
            ( 4.0,  0.9189, 0.00721, -0.1022),
            ( 4.5,  0.9730, 0.00748, -0.1018),
            ( 5.0,  1.0265, 0.00778, -0.1013),
            ( 5.5,  1.0794, 0.00811, -0.1008),
            ( 6.0,  1.1310, 0.00853, -0.1000),
            ( 6.5,  1.1812, 0.00903, -0.0990),
            ( 7.0,  1.2282, 0.00976, -0.0976),
            ( 7.5,  1.2718, 0.01069, -0.0956),
            ( 8.0,  1.3098, 0.01197, -0.0927),
            ( 8.5,  1.3449, 0.01334, -0.0894),
            ( 9.0,  1.3780, 0.01465, -0.0858),
            ( 9.5,  1.4062, 0.01593, -0.0813),
            (10.0,  1.4367, 0.01714, -0.0774),
            (10.5,  1.4654, 0.01848, -0.0735),
            (11.0,  1.4934, 0.01993, -0.0697),
            (11.5,  1.5182, 0.02164, -0.0660),
            (12.0,  1.5410, 0.02358, -0.0623),
            (12.5,  1.5612, 0.02587, -0.0588),
            (13.0,  1.5810, 0.02832, -0.0558),
            (13.5,  1.5913, 0.03175, -0.0526),
            (14.0,  1.5997, 0.03561, -0.0499),
        ]
    },
    'NACA_23012': {
        're': 1.0e6,
        'polars': [
            (-4.0, -0.2920, 0.00932, -0.0166),
            (-3.5, -0.2397, 0.00895, -0.0154),
            (-3.0, -0.1863, 0.00871, -0.0145),
            (-2.5, -0.1325, 0.00854, -0.0137),
            (-1.5, -0.0241, 0.00834, -0.0125),
            (-1.0,  0.0231, 0.00713, -0.0113),
            (-0.5,  0.0714, 0.00639, -0.0099),
            ( 0.0,  0.1226, 0.00620, -0.0088),
            ( 1.0,  0.2264, 0.00635, -0.0064),
            ( 1.5,  0.2775, 0.00660, -0.0048),
            ( 2.0,  0.3310, 0.00692, -0.0039),
            ( 2.5,  0.3872, 0.00723, -0.0037),
            ( 3.0,  0.4492, 0.00757, -0.0049),
            ( 3.5,  0.5176, 0.00788, -0.0075),
            ( 4.0,  0.5885, 0.00820, -0.0108),
            ( 4.5,  0.6588, 0.00851, -0.0140),
            ( 5.0,  0.7304, 0.00882, -0.0175),
            ( 5.5,  0.8011, 0.00915, -0.0208),
            ( 6.0,  0.8560, 0.00940, -0.0208),
            ( 6.5,  0.9023, 0.00968, -0.0189),
            ( 7.0,  0.9481, 0.01003, -0.0169),
            ( 7.5,  0.9950, 0.01035, -0.0152),
            ( 8.0,  1.0423, 0.01070, -0.0135),
            ( 8.5,  1.0898, 0.01109, -0.0119),
            ( 9.0,  1.1377, 0.01152, -0.0105),
            ( 9.5,  1.1862, 0.01200, -0.0092),
            (10.0,  1.2348, 0.01253, -0.0081),
            (10.5,  1.2828, 0.01314, -0.0070),
            (11.5,  1.3741, 0.01475, -0.0043),
            (12.0,  1.4158, 0.01583, -0.0025),
            (12.5,  1.4530, 0.01718, -0.0002),
            (13.0,  1.4860, 0.01870,  0.0026),
            (13.5,  1.5110, 0.02038,  0.0065),
            (14.0,  1.5306, 0.02223,  0.0107),
        ]
    },
    'NACA_63215': {
        're': 1.0e6,
        'polars': [
            (-3.5, -0.4117, 0.00721,  0.0033),
            (-3.0, -0.3432, 0.00648,  0.0002),
            (-2.5, -0.2732, 0.00582, -0.0031),
            (-2.0, -0.2110, 0.00530, -0.0045),
            (-1.5, -0.1548, 0.00487, -0.0043),
            (-1.0, -0.1028, 0.00456, -0.0030),
            (-0.5, -0.0515, 0.00435, -0.0015),
            ( 0.0,  0.0000, 0.00429,  0.0000),
            ( 0.5,  0.0515, 0.00435,  0.0015),
            ( 1.5,  0.1548, 0.00487,  0.0043),
            ( 2.0,  0.2110, 0.00530,  0.0045),
            ( 2.5,  0.2732, 0.00582,  0.0031),
            ( 3.0,  0.3432, 0.00648, -0.0002),
            ( 3.5,  0.4117, 0.00721, -0.0033),
            ( 4.0,  0.4820, 0.00793, -0.0068),
            ( 4.5,  0.5392, 0.00853, -0.0073),
            ( 5.0,  0.5868, 0.00905, -0.0057),
            ( 5.5,  0.6346, 0.00960, -0.0040),
            ( 6.0,  0.6829, 0.01018, -0.0024),
            ( 6.5,  0.7290, 0.01125, -0.0004),
            ( 7.0,  0.7795, 0.01163,  0.0007),
            ( 7.5,  0.8283, 0.01247,  0.0021),
            ( 8.0,  0.8693, 0.01480,  0.0046),
            ( 8.5,  0.9177, 0.01585,  0.0058),
            ( 9.0,  0.9650, 0.01709,  0.0071),
            ( 9.5,  1.0115, 0.01840,  0.0084),
            (10.0,  1.0582, 0.01954,  0.0095),
            (10.5,  1.1038, 0.02071,  0.0107),
            (11.0,  1.0878, 0.03070,  0.0174),
            (11.5,  1.0895, 0.03597,  0.0212),
            (12.0,  1.0567, 0.04345,  0.0239),
            (12.5,  0.9994, 0.06128,  0.0097),
            (13.5,  0.9221, 0.09995, -0.0153),
            (14.0,  0.8666, 0.12175, -0.0252),
        ]
    },
}

# Build combined DataFrame
rows = []
for name, info in raw_data.items():
    for (alpha, cl, cd, cm) in info['polars']:
        rows.append({
            'airfoil': name,
            'alpha':   alpha,
            'Cl':      cl,
            'Cd':      cd,
            'Cm':      cm,
            'Re':      info['re'],
        })

df = pd.DataFrame(rows)

# Encode airfoil name as integer
le = LabelEncoder()
df['airfoil_enc'] = le.fit_transform(df['airfoil'])

print(f"Dataset shape : {df.shape}")
print(f"Airfoils      : {df['airfoil'].unique().tolist()}")
print(f"Reynolds No.  : {df['Re'].unique()} (consistent across all)")
print(f"\nFirst 5 rows:")
print(df.head().to_string(index=False))

### CELL 3: Dataset Statistics per Airfoil

In [ ]:
print("=" * 60)
print("Dataset Statistics — Re = 1.0 x 10^6 (all airfoils)")
print("=" * 60)

stats = []
for name, grp in df.groupby('airfoil'):
    ld = grp['Cl'] / grp['Cd']
    stats.append({
        'Airfoil':   name,
        'N points':  len(grp),
        'Cl_max':    round(grp['Cl'].max(), 4),
        'Cd_min':    round(grp['Cd'].min(), 5),
        'max L/D':   round(ld.max(), 2),
        'α @ max L/D': round(float(grp.loc[ld.idxmax(), 'alpha']), 1),
    })

stats_df = pd.DataFrame(stats)
print(stats_df.to_string(index=False))

### CELL 4: Train/Test Split — Group by Airfoil

In [ ]:
# Hold out ONE complete airfoil as test set (unseen airfoil approach)
# This tests whether the model generalises beyond memorisation

X      = df[['alpha', 'Cl', 'Re', 'airfoil_enc']].values
y      = df['Cd'].values
groups = df['airfoil'].values

gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

train_af = df['airfoil'].iloc[train_idx].unique()
test_af  = df['airfoil'].iloc[test_idx].unique()

print(f"Training airfoils ({len(train_af)}): {list(train_af)}")
print(f"Test airfoil     ({len(test_af)}): {list(test_af)}")
print(f"Train samples: {len(X_train)}  |  Test samples: {len(X_test)}")

### CELL 5: Model 1 — Linear Regression (Baseline)

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

mae_lr = mean_absolute_error(y_test, y_pred_lr)
r2_lr  = r2_score(y_test, y_pred_lr)

print("=" * 45)
print("LINEAR REGRESSION — Baseline")
print("=" * 45)
print(f"  MAE : {mae_lr:.6f}")
print(f"  R²  : {r2_lr:.4f}")
print("Note: LR assumes linear relationship between features")
print("and Cd — expected to underperform on nonlinear regions.")

### CELL 6: Model 2 — Random Forest Regressor

In [ ]:
rf = RandomForestRegressor(n_estimators=200, max_depth=8,
                            random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

mae_rf = mean_absolute_error(y_test, y_pred_rf)
r2_rf  = r2_score(y_test, y_pred_rf)

print("=" * 45)
print("RANDOM FOREST REGRESSOR — Main Model")
print("=" * 45)
print(f"  MAE : {mae_rf:.6f}")
print(f"  R²  : {r2_rf:.4f}")

feat_names  = ['alpha', 'Cl', 'Re', 'airfoil_enc']
importances = rf.feature_importances_
print(f"\n  Feature importances:")
for fn, fi in sorted(zip(feat_names, importances), key=lambda x: -x[1]):
    print(f"    {fn:15s}: {fi:.4f}")

### CELL 7: Predicted vs Actual Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Predicted vs Actual Cd — Test Airfoil (Unseen)',
             fontsize=13, fontweight='bold')

for ax, y_pred, title, mae, r2, color in [
    (axes[0], y_pred_lr, 'Linear Regression (Baseline)', mae_lr, r2_lr, '#378ADD'),
    (axes[1], y_pred_rf, 'Random Forest (Main)',          mae_rf, r2_rf, '#D85A30'),
]:
    lo = min(y_test.min(), y_pred.min()) * 0.95
    hi = max(y_test.max(), y_pred.max()) * 1.05
    ax.scatter(y_test, y_pred, color=color, alpha=0.75,
               edgecolors='white', linewidth=0.5, s=65)
    ax.plot([lo, hi], [lo, hi], 'k--', linewidth=1.2, label='Perfect fit')
    ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
    ax.set_xlabel('Actual Cd  (XFLR)', fontsize=11)
    ax.set_ylabel('Predicted Cd', fontsize=11)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.text(0.05, 0.93,
            f'MAE = {mae:.5f}\nR²  = {r2:.4f}',
            transform=ax.transAxes, fontsize=10,
            verticalalignment='top',
            bbox=dict(boxstyle='round,pad=0.4',
                      facecolor='#f5f5f5', alpha=0.85))
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_facecolor('#fafafa')

plt.tight_layout()
plt.savefig('predicted_vs_actual.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: predicted_vs_actual.png")

### CELL 8: ML-Based Airfoil Ranking at Cruise

In [ ]:
# Cruise Cl = 0.6
# Justification: Trainer/GA aircraft cruising at moderate speed
# typically operate at Cl = 0.4–0.7 in level flight.
# Cl = 0.6 is selected as a representative mid-cruise value.

CRUISE_CL = 0.6
print(f"Cruise Cl = {CRUISE_CL}  (trainer/GA mid-cruise estimate)")
print("-" * 65)

COLORS = {
    'NACA_64A210': '#BA7517',
    'NACA_2412':   '#378ADD',
    'NACA_4412':   '#D85A30',
    'NACA_23012':  '#1D9E75',
    'NACA_63215':  '#993556',
}

ranking_rows = []
for name, info in raw_data.items():
    enc = le.transform([name])[0]
    re  = info['re']
    sub = df[df['airfoil'] == name].copy()

    # Closest data point to cruise Cl
    closest   = sub.iloc[(sub['Cl'] - CRUISE_CL).abs().argsort()[:1]]
    alpha_c   = float(closest['alpha'].values[0])
    cl_actual = float(closest['Cl'].values[0])
    cd_xflr   = float(closest['Cd'].values[0])

    # ML predictions
    X_q       = np.array([[alpha_c, CRUISE_CL, re, enc]])
    cd_rf     = float(rf.predict(X_q)[0])
    cd_lr     = float(lr.predict(X_q)[0])

    ld_xflr = CRUISE_CL / cd_xflr if cd_xflr > 0 else 0
    ld_rf   = CRUISE_CL / cd_rf   if cd_rf   > 0 else 0

    ranking_rows.append({
        'Airfoil':      name,
        'α cruise (°)': alpha_c,
        'Cl actual':    round(cl_actual, 4),
        'Cd XFLR':      round(cd_xflr, 5),
        'Cd RF pred':   round(cd_rf,   5),
        'Cd LR pred':   round(cd_lr,   5),
        'L/D XFLR':     round(ld_xflr, 2),
        'L/D RF pred':  round(ld_rf,   2),
    })

rank_df = pd.DataFrame(ranking_rows)
rank_df['XFLR Rank'] = rank_df['L/D XFLR'].rank(ascending=False).astype(int)
rank_df['RF Rank']   = rank_df['L/D RF pred'].rank(ascending=False).astype(int)
rank_df = rank_df.sort_values('L/D XFLR', ascending=False).reset_index(drop=True)

print(rank_df[['Airfoil','α cruise (°)','Cd XFLR','Cd RF pred',
               'L/D XFLR','L/D RF pred','XFLR Rank','RF Rank']].to_string(index=False))

### CELL 9: Ranking Bar Chart

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle(f'Airfoil L/D Ranking at Cruise Cl = {CRUISE_CL}  '
             f'(Re = 1.0 × 10⁶, all airfoils)',
             fontsize=12, fontweight='bold')

for ax, col, title in [
    (axes[0], 'L/D XFLR',    'XFLR Direct'),
    (axes[1], 'L/D RF pred', 'Random Forest Prediction'),
]:
    bar_colors = [COLORS[a] for a in rank_df['Airfoil']]
    bars = ax.barh(rank_df['Airfoil'], rank_df[col],
                   color=bar_colors, edgecolor='white',
                   linewidth=0.5, height=0.55)
    for bar, val in zip(bars, rank_df[col]):
        ax.text(bar.get_width() + 0.5,
                bar.get_y() + bar.get_height() / 2,
                f'{val:.1f}', va='center', fontsize=10)
    ax.set_xlabel('L/D ratio', fontsize=11)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)
    ax.set_facecolor('#fafafa')
    ax.invert_yaxis()

plt.tight_layout()
plt.savefig('airfoil_ranking.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: airfoil_ranking.png")

### CELL 10: Summary Table + Interpretation

In [ ]:
print("\n" + "=" * 55)
print("MODEL PERFORMANCE SUMMARY")
print("=" * 55)
print(f"{'Model':<28} {'MAE':>10} {'R²':>10}")
print("-" * 55)
print(f"{'Linear Regression':<28} {mae_lr:>10.6f} {r2_lr:>10.4f}")
print(f"{'Random Forest':<28} {mae_rf:>10.6f} {r2_rf:>10.4f}")
print("=" * 55)

print("""
KEY OBSERVATIONS:
─────────────────────────────────────────────────────
1. FAIR COMPARISON: All 5 airfoils now run at Re = 1.0×10⁶
   — eliminates Reynolds number bias from ranking.

2. RANDOM FOREST outperforms Linear Regression on both
   MAE and R² because Cd vs alpha is nonlinear, especially
   near stall and laminar-turbulent transition regions.

3. NACA 4412 ranks #1 in L/D at cruise. Its 4% camber
   generates high lift at low drag in this Re range.

4. NACA 63-215 has the lowest Cd in the low-alpha bucket
   (laminar bucket) but drops sharply post-transition.

5. NACA 23012 shows the best Cm stability (near zero Cm
   across cruise range) — important for trim & handling.

LIMITATIONS:
─────────────────────────────────────────────────────
- ML learns from XFLR data only; it does not add physics.
- Predictions outside the trained alpha/Cl range are
  unreliable — do not extrapolate.
- Dataset is small (~160 points); more Re sweep data would
  improve model generalisation.
- XFLR itself is based on panel + boundary layer methods
  and may deviate from real wind tunnel results at high AoA.
""")

print("All outputs saved:")
print("  predicted_vs_actual.png")
print("  airfoil_ranking.png")